# Глава 3. Линейная регрессия и метод наименьших квадратов

Код к главе 3 книги 1: формулы МНК для одного признака, нормальная система, геометрия
проекции, мультиколлинеарность и сингулярное разложение, полиномиальная регрессия
(листинги 3.1 и 3.2).

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markpolyak/ml-textbook-code/blob/main/book1/ch03_linreg.ipynb)

In [ ]:
# В Colab ставим пакет mlbook с данными книги (локально: pip install -e . в корне репозитория)
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/markpolyak/ml-textbook-code

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mlbook import latency

## Один признак

Выборка из трёх точек $(1; 1)$, $(2; 3)$, $(3; 2)$. Наклон и свободный член — по формулам
(3.1) и (3.2): $\theta_1 = \sum (x_i - \bar x)(y_i - \bar y) / \sum (x_i - \bar x)^2$,
$\theta_0 = \bar y - \theta_1 \bar x$.

In [ ]:
x = np.array([1.0, 2.0, 3.0])
y = np.array([1.0, 3.0, 2.0])
theta1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
theta0 = y.mean() - theta1 * x.mean()
pred = theta0 + theta1 * x
print("θ0 =", theta0, " θ1 =", theta1)
print("прогнозы:", pred, " остатки:", pred - y, " сумма остатков:", np.sum(pred - y))
print("Q =", np.mean((pred - y) ** 2))

Линии уровня $Q(\theta_0, \theta_1)$ — вложенные эллипсы с центром в решении (рис. 3.1).

In [ ]:
T0, T1 = np.meshgrid(np.linspace(-1, 3, 300), np.linspace(-0.5, 1.5, 300))
Q = ((T0[..., None] + T1[..., None] * x - y) ** 2).mean(axis=-1)

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.8))
xs = np.linspace(0, 4, 2)
a.plot(xs, theta0 + theta1 * xs, label="$g(x) = 1 + 0{,}5x$")
a.vlines(x, y, pred, colors="C1", linestyles="--", label="остатки")
a.scatter(x, y, color="black", zorder=3)
a.set_title("а) прямая и остатки")
a.legend()
cs = b.contour(T0, T1, Q, levels=[0.6, 1.2, 2, 3, 4.5])
b.clabel(cs)
b.plot(theta0, theta1, "o", color="C1")
b.set_xlabel(r"$\theta_0$")
b.set_ylabel(r"$\theta_1$")
b.set_title(r"б) линии уровня $Q(\theta_0, \theta_1)$")
plt.show()

## Нормальная система

Та же задача в матричной записи: $\mathbf{F}^{\mathrm{T}} \mathbf{F} \boldsymbol{\theta} =
\mathbf{F}^{\mathrm{T}} \mathbf{y}$. Обратную матрицу считаем здесь только для сверки
с текстом; на практике систему решают, не обращая матрицу.

In [ ]:
F = np.column_stack([np.ones(3), x])
A, b = F.T @ F, F.T @ y
print("FᵀF =\n", A, "\nFᵀy =", b, "\nопределитель:", round(np.linalg.det(A), 6))
print("(FᵀF)⁻¹ · 6 =\n", (np.linalg.inv(A) * 6).round(6))
print("θ* =", np.linalg.solve(A, b))

Листинг 3.1: то же решение функцией `lstsq` (через сингулярное разложение) и классом
`LinearRegression`.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

x = np.array([[1.0], [2.0], [3.0]])        # один признак
y = np.array([1.0, 3.0, 2.0])
F = np.hstack([np.ones((3, 1)), x])        # столбец f0 = 1
theta = np.linalg.lstsq(F, y, rcond=None)[0]
print(theta)                               # [1.  0.5]

model = LinearRegression().fit(x, y)       # f0 добавляется сам
print(model.intercept_, model.coef_)       # 1.0 [0.5]

## Геометрический смысл

Вектор остатков перпендикулярен каждому столбцу $\mathbf{F}$, а прогнозы — проекция
вектора ответов: $\mathbf{F}\boldsymbol{\theta}^* = \mathbf{P}\mathbf{y}$,
$\mathbf{P} = \mathbf{F}(\mathbf{F}^{\mathrm{T}}\mathbf{F})^{-1}\mathbf{F}^{\mathrm{T}}$.

In [ ]:
r = y - F @ theta
print("остатки:", r, " Fᵀ(y − Fθ*):", np.abs((F.T @ r).round(12)))
P = F @ np.linalg.inv(F.T @ F) @ F.T
print("P =\n", P.round(3))
print("Py =", P @ y, " Fθ* =", F @ theta)
print("P² = P:", np.allclose(P @ P, P))    # повторная проекция ничего не меняет

## Мультиколлинеарность

Два почти совпадающих признака, свободного члена нет. Определитель $\mathbf{F}^{\mathrm{T}}
\mathbf{F}$ мал по сравнению с элементами, элементы обратной матрицы — около 290.

In [ ]:
F = np.array([[1, 1], [2, 2], [3, 3.1]])
print("FᵀF =\n", F.T @ F, "\nопределитель:", round(np.linalg.det(F.T @ F), 4))
print("(FᵀF)⁻¹ =\n", np.linalg.inv(F.T @ F).round(1))

Малое изменение последнего ответа полностью меняет веса, но почти не меняет прогнозы
на обучающих объектах; прогнозы для нового объекта $(2;\ 3)$ расходятся (табл. 3.1).

In [ ]:
for y3 in (6.0, 6.1, 6.2):
    y = np.array([2, 4, y3])
    theta = np.linalg.solve(F.T @ F, F.T @ y)
    print(f"y3 = {y3}: θ* = {theta.round(6)}, прогнозы {(F @ theta).round(6)},",
          f"прогноз для (2; 3): {np.array([2, 3]) @ theta:.4g}")

Линии уровня $Q(\theta_1, \theta_2)$ — длинный «овраг» вдоль $\theta_1 + \theta_2 \approx 2$;
точки — три решения из таблицы (рис. 3.3).

In [ ]:
T1, T2 = np.meshgrid(np.linspace(-1, 3, 400), np.linspace(-1, 3, 400))
R = T1[..., None] * F[:, 0] + T2[..., None] * F[:, 1] - np.array([2, 4, 6.1])
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.contour(T1, T2, (R ** 2).mean(axis=-1), levels=[0.003, 0.03, 0.3, 1, 3])
ax.plot([2, 1, 0], [0, 1, 2], "o", color="C1")
ax.set_xlabel(r"$\theta_1$")
ax.set_ylabel(r"$\theta_2$")
ax.set_aspect("equal")
ax.grid(True)
plt.show()

**Сингулярное разложение.** Буквы в NumPy другие, чем в книге: `np.linalg.svd(F)`
возвращает матрицу $\mathbf{V}$ под именем `U`, вектор сингулярных чисел и матрицу
$\mathbf{U}^{\mathrm{T}}$. Малое $s_2$ — признак почти линейной зависимости столбцов,
$\mathbf{u}_2$ — почти разность двух признаков.

In [ ]:
V, s, Ut = np.linalg.svd(F, full_matrices=False)
print("сингулярные числа:", s.round(3))
print("u2 =", Ut[1].round(2), "(знак у сингулярного вектора произволен)")
print("κ(FᵀF) =", round(s[0] ** 2 / s[1] ** 2), " np.linalg.cond:", round(np.linalg.cond(F.T @ F)))

Решение через SVD: каждая проекция $\mathbf{v}_j^{\mathrm{T}}\mathbf{y}$ делится на $s_j$.

In [ ]:
y = np.array([2, 4, 6.1])
theta = Ut.T @ ((V.T @ y) / s)
print(theta.round(6))

**Псевдообратная матрица** даёт решение и при точной линейной зависимости признаков
(задача 3): второй признак — первый в других единицах, $f_2 = 100 f_1$. Решение наименьшей
длины распределяет вес так, что $\theta_2 = 100\,\theta_1$.

In [ ]:
F2 = np.array([[1, 100], [2, 200], [3, 300.0]])
print("определитель FᵀF:", round(np.linalg.det(F2.T @ F2), 6))
theta = np.linalg.pinv(F2) @ np.array([2, 4, 6.0])
print("θ =", theta, " θ2/θ1 =", round(theta[1] / theta[0], 6))

## Полиномиальная регрессия

Время отклика сервиса в зависимости от нагрузки: 12 точек, многочлены степеней 1, 2 и 9
по стандартизованному признаку (рис. 3.4).

In [ ]:
x, y = latency.make()
z = (x - x.mean()) / x.std()
xs = np.linspace(1, 10, 400)
zs = (xs - x.mean()) / x.std()

fig, ax = plt.subplots(figsize=(6, 4))
for d in (1, 2, 9):
    theta = np.linalg.lstsq(np.vander(z, d + 1, increasing=True), y, rcond=None)[0]
    ax.plot(xs, np.vander(zs, d + 1, increasing=True) @ theta, label=f"степень {d}")
ax.scatter(x, y, color="black", zorder=3)
ax.set_ylim(-10, 200)
ax.set_xlabel("нагрузка, тыс. запросов в секунду")
ax.set_ylabel("время отклика, мс")
ax.legend()
plt.show()

Столбцы $x^4$ и $x^5$ почти пропорциональны, и матрица $\mathbf{F}^{\mathrm{T}}\mathbf{F}$
для многочлена пятой степени плохо обусловлена; стандартизация признака это исправляет.

In [ ]:
for name, t in [("x", x), ("стандартизованный x", z)]:
    Fp = np.vander(t, 6, increasing=True)
    print(f"{name}: κ(FᵀF) = {np.linalg.cond(Fp.T @ Fp):.1e}")

Листинг 3.2: стандартизация, степени признака и линейная регрессия в одном конвейере.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

model = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression())

Обучим конвейер на выборке побольше — 200 измерений при случайной нагрузке.

In [ ]:
df = latency.load()
X, y = df[["нагрузка"]], df["время"]
model.fit(X, y)
print(model[1].get_feature_names_out(), model[-1].coef_.round(2), round(model[-1].intercept_, 2))
new = pd.DataFrame({"нагрузка": [5.0, 12.0]})
print("прогноз при нагрузке 5 и 12:", model.predict(new).round(1),
      " истинное время:", latency.response_time([5, 12]))

При нагрузке 12 модель экстраполирует — за пределы отрезка $[1;\ 10]$, на котором она
обучена. Здесь это сошлось, потому что вид зависимости угадан; в общем случае экстраполяция
ничем не проверена.

## Задачи

**1–2.** Точки $(0; 1)$, $(1; 1)$, $(2; 4)$: формулы для одного признака и нормальная система.

In [ ]:
x = np.array([0.0, 1, 2])
y = np.array([1.0, 1, 4])
t1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
t0 = y.mean() - t1 * x.mean()
print("θ0 =", t0, " θ1 =", t1, " остатки:", t0 + t1 * x - y)
F = np.column_stack([np.ones(3), x])
print("FᵀF =", (F.T @ F).tolist(), " Fᵀy =", F.T @ y, " θ* =", np.linalg.solve(F.T @ F, F.T @ y))